# NEXRAD Level 2 Format Demo

This notebook demonstrates reading three different NEXRAD Level 2 file variants with xradar:

1. **Modern MSG 31** (post-2008) - dual-pol moments, super-resolution
2. **Legacy MSG 1** (pre-2008) - single-pol, lower resolution
3. **Chunk files** (S/I/E) - real-time LDM streaming format

Each section shows the data structure and a PPI plot of reflectivity.

In [ ]:
import cmweather  # noqa: registers colormaps
import gzip
import os
import shutil
import tarfile
import tempfile

from open_radar_data import DATASETS

import xradar as xd

# Check if the Rust backend is available
from xradar.io.backends.nexrad_level2 import _HAS_RUST
print(f"Rust NEXRAD backend: {'enabled' if _HAS_RUST else 'disabled (using Python)'}")

---
## 1. Modern MSG 31 Format (KATX, VCP-11)

Post-2008 NEXRAD data uses Message Type 31 with dual-polarization moments
(REF, VEL, SW, ZDR, PHI, RHO) and 0.5-degree super-resolution azimuth spacing
(720 rays per sweep). This file is uncompressed.

In [ ]:
f_modern = DATASETS.fetch("KATX20130717_195021_V06")
dtree_modern = xd.io.open_nexradlevel2_datatree(f_modern)
display(dtree_modern)

In [ ]:
# Root attributes: VCP info, site coordinates
print("Root attributes:")
for k, v in dtree_modern.ds.attrs.items():
    print(f"  {k}: {v}")
print(f"\nStation: ({float(dtree_modern.ds.latitude):.4f}N, {float(dtree_modern.ds.longitude):.4f}E)")
print(f"Sweeps: {len(dtree_modern.match('sweep_*'))}")

In [ ]:
# Sweep 0 dataset structure
ds0 = dtree_modern["sweep_0"].ds
display(ds0)

In [ ]:
# PPI plot of reflectivity (Range vs Azimuth)
dtree_modern["sweep_0"].ds.DBZH.plot(cmap="ChaseSpectral", vmin=-10, vmax=60)

---
## 2. Legacy MSG 1 Format (KLIX, 2005)

Pre-2008 NEXRAD data uses Message Type 1 with single-polarization moments
(REF, VEL, SW) and 1-degree azimuth spacing (360 rays per sweep).
Angles use a different encoding (scaled by 180/32768).
This file from Hurricane Katrina (2005-08-28) is gzip-compressed on disk.

In [ ]:
# Decompress the gzip file
f_gz = DATASETS.fetch("KLIX20050828_180149.gz")
f_legacy = tempfile.NamedTemporaryFile(delete=False, suffix="_KLIX").name
with gzip.open(f_gz) as fin, open(f_legacy, "wb") as fout:
    shutil.copyfileobj(fin, fout)

dtree_legacy = xd.io.open_nexradlevel2_datatree(f_legacy)
display(dtree_legacy)

In [ ]:
print("Root attributes:")
for k, v in dtree_legacy.ds.attrs.items():
    print(f"  {k}: {v}")
print(f"\nSweeps: {len(dtree_legacy.match('sweep_*'))}")

In [ ]:
# Sweep 0 - note: only REF (no dual-pol moments)
ds0_legacy = dtree_legacy["sweep_0"].ds
display(ds0_legacy)

In [ ]:
# PPI of Hurricane Katrina reflectivity
dtree_legacy["sweep_0"].ds.DBZH.plot(cmap="ChaseSpectral", vmin=-10, vmax=60)

In [ ]:
os.unlink(f_legacy)

---
## 3. Chunk Files (KLOT, VCP-35)

Real-time NEXRAD data arrives via LDM as chunk files:
- **S file**: volume scan start (contains the volume header + metadata)
- **I files**: intermediate chunks (BZ2-compressed radar data)
- **E file**: end of volume

xradar concatenates these automatically when you pass a list of paths.
This example has 55 chunks from KLOT (Chicago/Romeoville).

In [ ]:
# Extract chunk files from archive
archive = DATASETS.fetch("nexrad_level2_chunks_KLOT.tar.gz")
tmpdir = tempfile.mkdtemp()
with tarfile.open(archive) as tar:
    tar.extractall(tmpdir, filter="data")

chunk_dir = os.path.join(tmpdir, "nexrad_chunks_KLOT")
chunk_paths = sorted(
    [os.path.join(chunk_dir, f) for f in os.listdir(chunk_dir)]
)
print(f"Chunk files: {len(chunk_paths)}")
print(f"First: {os.path.basename(chunk_paths[0])}")
print(f"Last:  {os.path.basename(chunk_paths[-1])}")

In [ ]:
# Pass the list of chunk paths directly
dtree_chunks = xd.io.open_nexradlevel2_datatree(chunk_paths)
display(dtree_chunks)

In [ ]:
print("Root attributes:")
for k, v in dtree_chunks.ds.attrs.items():
    print(f"  {k}: {v}")
print(f"\nSweeps: {len(dtree_chunks.match('sweep_*'))}")

In [ ]:
# Sweep 0 from reassembled chunk volume
ds0_chunks = dtree_chunks["sweep_0"].ds
display(ds0_chunks)

In [ ]:
# PPI from chunk-reassembled volume
dtree_chunks["sweep_0"].ds.DBZH.plot(cmap="ChaseSpectral", vmin=-10, vmax=60)

In [ ]:
shutil.rmtree(tmpdir)

---
## Summary

| Format | File | VCP | Sweeps | Moments | Rays/Sweep |
|--------|------|-----|--------|---------|------------|
| Modern MSG 31 | KATX (2013) | VCP-11 | 16 | REF, VEL, SW, ZDR, PHI, RHO | 720 |
| Legacy MSG 1 | KLIX (2005) | VCP-0 | 16 | REF, VEL, SW | 367 |
| Chunk S/I/E | KLOT (2026) | VCP-35 | 12 | REF, VEL, SW, ZDR, PHI, RHO, CFP | 720 |

All three formats are read through the same `open_nexradlevel2_datatree()` API.
When the Rust extension is installed, parsing is 3-6x faster.